Based on:
- Rogers, E. M. (1995). *Diffusion of Innovations* (4th ed.). Free Press.
- https://albertoacerbi.github.io/IBM-cultevo/social-network-structure.html

With added comments and some modifications by AB.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

np.random.seed(42)

# 0. Primer on network

Networks are a popular tool to visualise relationships between different actors. For example, co-authorship networks visualise which authors frequently publish together, ecological networks demonstrate trophic interactions between species, and kinship networks show how individuals are related to each other.

All networks are comprised of at least two components:
- **Nodes** (also called vertices): represent actors
- **Edges** (also called ties): represent the existence of a relationship between two nodes (e.g. friendship, kinship, cooperation)

In Python, we use the `networkx` library to create, manipulate, and analyse networks.

## Network from matrix

One way to represent a network is an **adjacency matrix** a square $N \times N$ matrix where entry $(i, j) = 1$ if individuals $i$ and $j$ have a relationship, and $0$ otherwise.

If relationships are **reciprocal** (e.g. friendships), the matrix is **symmetric**: $A_{ij} = A_{ji}$. If not (e.g. directed donations), the matrix can be asymmetric.

In [ ]:
A = np.array([
    [0, 1, 1, 0],
    [0, 0, 1, 0],
    [1, 0, 0, 1],
    [0, 0, 1, 0]
])

A

In [ ]:
# Check symmetry
is_symmetric = np.all(A == A.T)
is_symmetric

We can convert an adjacency matrix directly into a `networkx` graph and plot it:

In [ ]:
G_directed = nx.from_numpy_array(A, create_using=nx.DiGraph)

fig, ax = plt.subplots(figsize=(4, 4))
pos = nx.spring_layout(G_directed, seed=42)
nx.draw_networkx(G_directed, pos=pos, ax=ax,
                 node_color="steelblue", node_size=800,
                 labels={i: chr(65+i) for i in range(4)},
                 font_color="white", font_size=12,
                 edge_color="grey", arrows=True,
                 connectionstyle="arc3,rad=0.2")
ax.set_title("Directed network (asymmetric)")
ax.axis("off")
plt.tight_layout()
plt.show()

For undirected (reciprocal) networks, we use a **symmetric** adjacency matrix. `networkx` handles this automatically when using `nx.Graph`

In [ ]:
# Make the matrix symmetric
A_sym = np.maximum(A, A.T)
print("Is symmetric:", np.array_equal(A_sym, A_sym.T))

# Undirected graph
G = nx.from_numpy_array(A_sym)

fig, ax = plt.subplots(figsize=(4, 4))
nx.draw_networkx(G, pos=pos, ax=ax,
                 node_color="steelblue", node_size=800,
                 labels={i: chr(65+i) for i in range(4)},
                 font_color="white", font_size=12,
                 edge_color="grey")
ax.set_title("Undirected network (symmetric)")
ax.axis("off")
plt.tight_layout()
plt.show()

## Generate network

Real-world networks vary widely in structure. `networkx` provides generators for several canonical network types. Below we create and visualise five of them, all with $N = 25$ nodes:

| Network type | Generator | Key property |
|---|---|---|
| Ring | `nx.cycle_graph` | Every node has exactly 2 neighbours |
| Lattice | `nx.grid_2d_graph` | Regular grid structure |
| Random | `nx.erdos_renyi_graph` | Edges placed with uniform probability |
| Scale-free | `nx.barabasi_albert_graph` | Few hubs, many low-degree nodes |
| Fully connected | `nx.complete_graph` | Every node connected to every other |

In [ ]:
N = 25
graph_types = {
    "Ring":            nx.cycle_graph(N),
    "Lattice":         nx.convert_node_labels_to_integers(nx.grid_2d_graph(5, 5)),
    "Random":          nx.erdos_renyi_graph(N, p=0.15, seed=42),
    "Scale-free":      nx.barabasi_albert_graph(N, m=2, seed=42),
    "Fully connected": nx.complete_graph(N),
}

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, (name, G_) in zip(axes, graph_types.items()):
    pos_ = nx.spring_layout(G_, seed=42)
    nx.draw_networkx(G_, pos=pos_, ax=ax,
                     node_color="#18889F", node_size=200,
                     edge_color="#33333B", with_labels=False,
                     width=1)
    ax.set_title(name)
    ax.axis("off")
plt.tight_layout()
plt.show()

We can also build our own network from scratch using an adjacency matrix with a friendship proportion $f$:
- $f = 1$: everyone is friends with everyone
- $f = 0$: no one is friends with anyone

In [ ]:
def create_network(N, f):
    """Create a random symmetric adjacency matrix.

    Parameters
    ----------
    N (int)  : number of individuals
    f (float): friendship proportion (0 = no edges, 1 = fully connected)

    Returns
    -------
    nx.Graph
    """
    A       = np.zeros((N, N), dtype=int)
    target  = round(((N**2 - N) / 2) * f)
    friends = 0

    while friends < target:
        i, j = np.random.choice(N, size=2, replace=False)
        if A[i, j] == 0:
            A[i, j] = A[j, i] = 1
            friends += 1

    return nx.from_numpy_array(A)


G = create_network(N=20, f=0.3)

fig, ax = plt.subplots(figsize=(5, 5))
pos = nx.spring_layout(G, seed=42)
nx.draw_networkx(G, pos=pos, ax=ax,
                 node_color="steelblue", node_size=400,
                 edge_color="grey", with_labels=True,
                 font_color="white")
ax.set_title(f"Custom random network (N=20, f=0.3)")
ax.axis("off")
plt.tight_layout()
plt.show()

## Node attributes

We can attach any information to nodes (age, group membership, behaviour, etc.) and use it to colour the network. This makes it easy to visualise how attributes relate to network position.

In [ ]:
G = create_network(N=20, f=0.4)
pos = nx.spring_layout(G, seed=42)
N   = G.number_of_nodes()

# Assign random ages
ages   = np.random.randint(18, 81, size=N)
nx.set_node_attributes(G, dict(enumerate(ages)), "age")

# Colour by age
cmap   = plt.cm.YlOrRd
norm   = plt.Normalize(18, 80)
colors = [cmap(norm(ages[i])) for i in G.nodes()]

fig, ax = plt.subplots(figsize=(6, 5))
nx.draw_networkx(G, pos=pos, ax=ax,
                 node_color=colors, node_size=500,
                 labels=dict(enumerate(ages)),
                 font_size=7, font_color="black",
                 edge_color="grey")
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
plt.colorbar(sm, ax=ax, label="Age")
ax.set_title("Network coloured by age")
ax.axis("off")
plt.tight_layout()
plt.show()

We can also vary **edge width** to represent relationship strength:

In [ ]:
# Assign random strength to each edge
strengths = {e: np.random.uniform(0, 1) for e in G.edges()}
nx.set_edge_attributes(G, strengths, "weight")

edge_widths = [G[u][v]["weight"] * 5 for u, v in G.edges()]

fig, ax = plt.subplots(figsize=(6, 5))
nx.draw_networkx(G, pos=pos, ax=ax,
                 node_color=colors, node_size=500,
                 with_labels=False,
                 edge_color="grey",
                 width=edge_widths)
ax.set_title("Edge width = relationship strength")
ax.axis("off")
plt.tight_layout()
plt.show()

## Network metrics

We distinguish between two levels of network description:
- **Population-level**: properties of the whole network
- **Node-level**: properties of individual nodes

### Population-level metrics

In [ ]:
G = create_network(N=30, f=0.25)

print(f"Number of nodes    : {G.number_of_nodes()}")
print(f"Number of edges    : {G.number_of_edges()}")
print(f"Edge density       : {nx.density(G):.3f}")
print(f"Clustering coeff.  : {nx.transitivity(G):.3f}")

if nx.is_connected(G):
    print(f"Diameter           : {nx.diameter(G)}")
    print(f"Avg. path length   : {nx.average_shortest_path_length(G):.3f}")
else:
    print("Graph not connected — diameter and avg. path length undefined")

**Diameter** is the longest shortest path between any two nodes. That is, the maximum number of steps needed to connect the two most distant individuals.

**Average path length** is the average number of steps between all pairs of nodes. It's a measure of how efficiently information can travel.

**Edge density** is the proportion of possible edges that are actually present.

**Clustering coefficient** (transitivity) is the probability that two neighbours of a node are also neighbours of each other ("my friends are friends with each other").

### <span style="color:red">Exercise:</span>

Try different network variable *N* and *f* to see how they impact the population-level metrics.

In [ ]:
G = create_network(N=?, f=?)

print(f"Number of nodes    : {G.number_of_nodes()}")
print(f"Number of edges    : {G.number_of_edges()}")
print(f"Edge density       : {nx.density(G):.3f}")
print(f"Clustering coeff.  : {nx.transitivity(G):.3f}")

if nx.is_connected(G):
    print(f"Diameter           : {nx.diameter(G)}")
    print(f"Avg. path length   : {nx.average_shortest_path_length(G):.3f}")
else:
    print("Graph not connected — diameter and avg. path length undefined")

### Node-level metrics

In [ ]:
G = create_network(N=10, f=0.55)

pos = nx.spring_layout(G, seed=1)

degree      = dict(G.degree())
closeness   = nx.closeness_centrality(G)
betweenness = nx.betweenness_centrality(G)
eigenvector = nx.eigenvector_centrality(G)

metrics = pd.DataFrame({
    "degree":      degree,
    "closeness":   closeness,
    "betweenness": betweenness,
    "eigenvector": eigenvector,
})
metrics.index.name = "node"
print(metrics.round(3).head(10))

**Degree centrality** is the number of edges connected to a node. High degree = many direct connections.

**Closeness centrality** is the inverse of the average shortest path to all other nodes. High closeness = information arrives quickly.

**Betweenness centrality** is the number of shortest paths between other nodes that pass through this node. High betweenness = gatekeeping / brokerage potential.

**Eigenvector centrality** is the a node is important if it is connected to other important nodes. High eigenvector = well-connected to well-connected individuals.

We can visualise all four metrics at once by colouring the same network four times:

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
cmap = plt.cm.YlOrRd

for ax, (metric_name, values) in zip(axes, metrics.items()):
    vals   = values.to_numpy()
    norm   = plt.Normalize(vals.min(), vals.max())
    colors = [cmap(norm(values[n])) for n in G.nodes()]

    nx.draw_networkx(G, pos=pos, ax=ax,
                     node_color=colors, node_size=300,
                     with_labels=False, edge_color="lightgrey")
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    plt.colorbar(sm, ax=ax, shrink=0.8)
    ax.set_title(metric_name.capitalize())
    ax.axis("off")

plt.suptitle("Node-level centrality metrics (yellow = low, red = high)", y=1.02)
plt.tight_layout()
plt.show()

Note that the four metrics do not always agree on which nodes are "important":
- A node can have high **degree** but low **betweenness** (many friends, but not a bridge)
- A node can have low **degree** but high **betweenness** (few friends, but sits on key paths)
- **Eigenvector** centrality rewards being connected to well-connected nodes, not just having many connections

### <span style="color:red">Exercise:</span>

Try different network variable *N* and *f* to see how they impact the node-level metrics.

In [ ]:
G = create_network(N=?, f=?)

pos = nx.spring_layout(G, seed=1)

degree      = dict(G.degree())
closeness   = nx.closeness_centrality(G)
betweenness = nx.betweenness_centrality(G)
eigenvector = nx.eigenvector_centrality(G)

metrics = pd.DataFrame({
    "degree":      degree,
    "closeness":   closeness,
    "betweenness": betweenness,
    "eigenvector": eigenvector,
})
metrics.index.name = "node"
print(metrics.round(3).head(10))

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
cmap = plt.cm.YlOrRd

for ax, (metric_name, values) in zip(axes, metrics.items()):
    vals   = values.to_numpy()
    norm   = plt.Normalize(vals.min(), vals.max())
    colors = [cmap(norm(values[n])) for n in G.nodes()]

    nx.draw_networkx(G, pos=pos, ax=ax,
                     node_color=colors, node_size=300,
                     with_labels=False, edge_color="lightgrey")
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    plt.colorbar(sm, ax=ax, shrink=0.8)
    ax.set_title(metric_name.capitalize())
    ax.axis("off")

plt.suptitle("Node-level centrality metrics (yellow = low, red = high)", y=1.02)
plt.tight_layout()
plt.show()

# A. The basic model

Each individual is in one of two states:

- **0 - Unaware**: has not yet adopted the innovation
- **1 - Adopter**: has adopted and can transmit to neighbours

Each round, every Adopter attempts to transmit the innovation to each of its neighbours independently with probability $p_t$. Adoption is **irreversible**: once adopted, always adopted.

We seed the model with a single random adopter and track the proportion of the population that has adopted over time.

In [ ]:
def diffusion_model(G, p_t, r_max, seed_node=None, sim=1):
    """Basic diffusion of innovations on a network.

    Parameters
    ----------
    G         (nx.Graph): social network
    p_t       (float)   : transmission probability per neighbour per round
    r_max     (int)     : number of rounds
    seed_node (int)     : node to seed (random if None)
    sim       (int)     : simulation ID

    Returns
    -------
    pd.DataFrame: columns time, proportion, sim
    """
    N      = G.number_of_nodes()
    adjm   = nx.to_numpy_array(G, dtype=bool)
    state  = np.zeros(N, dtype=int)   # 0 = unaware, 1 = adopter

    # Seed one node
    if seed_node is None:
        seed_node = np.random.randint(N)
    state[seed_node] = 1

    rows = []
    for r in range(r_max):
        new_state = state.copy()
        for i in np.where(state == 1)[0]:          # loop over current adopters
            neighbours = np.where(adjm[i])[0]
            for j in neighbours:
                if state[j] == 0:                  # only transmit to unaware
                    if np.random.uniform() < p_t:
                        new_state[j] = 1
        state = new_state
        rows.append({
            "time":       r + 1,
            "proportion": np.sum(state) / N,
            "sim":        sim
        })

    return pd.DataFrame(rows)

Let us run the model on a simple random network and observe the S-shaped diffusion curve:

In [ ]:
N  = 200
G  = nx.watts_strogatz_graph(n=N, k=6, p=0.1, seed=42)

res = diffusion_model(G=G, p_t=0.3, r_max=30)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(res["time"], res["proportion"])
ax.set_ylim(0, 1.05)
ax.set_xlabel("Time")
ax.set_ylabel("Proportion of adopters")
ax.set_title("Basic diffusion of innovations")
plt.tight_layout()
plt.show()

The S-shaped curve is the hallmark of diffusion processes: slow initial spread (few adopters, few transmission opportunities), rapid acceleration (many adopters), then deceleration as most of the population has already adopted.

### <span style="color:red">Exercise:</span>

Test the impact of populatoin size on this shape. Do you see any different with smaller/larger *N*?

In [ ]:
N  = ?
G  = nx.watts_strogatz_graph(n=N, k=6, p=0.1, seed=42)

res = diffusion_model(G=G, p_t=0.3, r_max=30)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(res["time"], res["proportion"])
ax.set_ylim(0, 1.05)
ax.set_xlabel("Time")
ax.set_ylabel("Proportion of adopters")
ax.set_title("Basic diffusion of innovations")
plt.tight_layout()
plt.show()

### Multiple replicates

We want to run multiple replicates and plot them together.

In [ ]:
reps = pd.concat([diffusion_model(G=G, p_t=0.3, r_max=30, sim=i)
                  for i in range(20)], ignore_index=True)

fig, ax = plt.subplots(figsize=(8, 4))
for sim, grp in reps.groupby("sim"):
    ax.plot(grp["time"], grp["proportion"], alpha=0.4, color="steelblue", linewidth=0.8)
ax.plot(reps.groupby("time")["proportion"].mean(), color="black",
        linewidth=2, label="Mean")
ax.set_ylim(0, 1.05)
ax.set_xlabel("Time")
ax.set_ylabel("Proportion of adopters")
ax.set_title("Diffusion of innovations")
ax.legend()
plt.tight_layout()
plt.show()

### <span style="color:red">Exercise:</span>

Explain why we want to run multiple run and average them.

# B. Effect of transmission probability $p_t$

The transmission probability $p_t$ controls how easily the innovation spreads between connected individuals. We compare several values across 20 replicates each:

In [ ]:
pt_values = [0.05, 0.1, 0.2, 0.4]
n_reps    = 20

results_pt = pd.concat([
    diffusion_model(G=G, p_t=pt, r_max=50, sim=i).assign(p_t=pt)
    for pt in pt_values
    for i in range(n_reps)
], ignore_index=True)

mean_pt = results_pt.groupby(["p_t", "time"])["proportion"].mean().reset_index()

fig, ax = plt.subplots(figsize=(9, 4))
for pt, grp in mean_pt.groupby("p_t"):
    ax.plot(grp["time"], grp["proportion"], label=f"$p_t={pt}$")
ax.set_ylim(0, 1.05)
ax.set_xlabel("Time")
ax.set_ylabel("Proportion of adopters")
ax.set_title("Effect of transmission probability")
ax.legend(title="$p_t$")
plt.tight_layout()
plt.show()

Higher $p_t$ accelerates spread and increases the final proportion of adopters. Below a certain threshold, the innovation fails to spread at all. This is the **epidemic threshold**, familiar from disease models.

# C. Adding abandonment

In Rogers' framework, not all adopters keep the innovation. We add an **abandonment probability** $p_a$: at each round, each adopter independently reverts to unaware with probability $p_a$.

We now have three parameters to play with: $p_t$, $p_a$, and network topology.

In [ ]:
def diffusion_model_abandonment(G, p_t, p_a, r_max, sim=1):
    """Diffusion of innovations with abandonment.

    Parameters
    ----------
    G    (nx.Graph): social network
    p_t  (float)   : transmission probability per neighbour per round
    p_a  (float)   : abandonment probability per round
    r_max (int)    : number of rounds
    sim  (int)     : simulation ID

    Returns
    -------
    pd.DataFrame: columns time, proportion, p_t, p_a, sim
    """
    N      = G.number_of_nodes()
    adjm   = nx.to_numpy_array(G, dtype=bool)
    state  = np.zeros(N, dtype=int)
    state[np.random.randint(N)] = 1

    rows = []
    for r in range(r_max):
        new_state = state.copy()
        for i in range(N):
            if state[i] == 1:
                # Abandonment
                if np.random.uniform() < p_a:
                    new_state[i] = 0
            else:
                # Transmission from adopting neighbours
                neighbours = np.where(adjm[i])[0]
                n_adopters = np.sum(state[neighbours])
                if n_adopters > 0:
                    # At least one adopting neighbour: attempt transmission
                    if np.random.uniform() < 1 - (1 - p_t) ** n_adopters:
                        new_state[i] = 1
        state = new_state
        rows.append({
            "time":       r + 1,
            "proportion": np.sum(state) / N,
            "p_t":        p_t,
            "p_a":        p_a,
            "sim":        sim
        })

    return pd.DataFrame(rows)

Notice the transmission step uses $1 - (1-p_t)^k$ where $k$ is the number of adopting neighbours. This is the probability that **at least one** of $k$ independent transmission attempts succeeds, more adopting neighbours means a higher chance of exposure.

In [ ]:
pa_values = [0.0, 0.05, 0.1, 0.2]
n_reps    = 20

results_pa = pd.concat([
    diffusion_model_abandonment(G=G, p_t=0.2, p_a=pa, r_max=100, sim=i).assign(p_a=pa)
    for pa in pa_values
    for i in range(n_reps)
], ignore_index=True)

mean_pa = results_pa.groupby(["p_a", "time"])["proportion"].mean().reset_index()

fig, ax = plt.subplots(figsize=(9, 4))
for pa, grp in mean_pa.groupby("p_a"):
    ax.plot(grp["time"], grp["proportion"], label=f"$p_a={pa}$")
ax.set_ylim(0, 1.05)
ax.set_xlabel("Time")
ax.set_ylabel("Proportion of adopters")
ax.set_title("Effect of abandonment probability ($p_t = 0.2$)")
ax.legend(title="$p_a$")
plt.tight_layout()
plt.show()

### <span style="color:red">Exercise:</span>

Can you explain why the line don't reach 1.0 for simulation with larger $p_a$ ? (hint: notice how the lines are not straight after plateauing)

<details>
  <summary>Answer</summary>
  With abandonment, the model reaches an <b>equilibrium</b> (or plateau) rather than full adoption.
  The proportion of adopters never reach 1.0 because individual keep losing the innovation while over are adopting it. This results in a jittery line that osciales around an equilibrium.
</details>


# D. Effect of network topology

One of the most powerful uses of network models is comparing how different network structures affect diffusion. We compare four canonical network types, all with $N=200$ and mean degree $\approx 6$:

- **Random** (Erdős–Rényi): edges placed uniformly at random
- **Small-world** (Watts–Strogatz): high clustering + short path lengths
- **Scale-free** (Barabási–Albert): a few highly connected hubs, many low-degree nodes
- **Lattice** (ring): high clustering, long path lengths

In [ ]:
N = 200
networks = {
    "random":      nx.erdos_renyi_graph(n=N, p=6/(N-1),    seed=42),
    "small-world": nx.watts_strogatz_graph(n=N, k=6, p=0.1, seed=42),
    "scale-free":  nx.barabasi_albert_graph(n=N, m=3,       seed=42),
    "lattice":     nx.watts_strogatz_graph(n=N, k=6, p=0,   seed=42),
}

# Print summary statistics
print(f"{'Network':<15} {'Mean degree':>12} {'Clustering':>12} {'Avg path':>10}")
print("-" * 52)
for name, G_ in networks.items():
    deg  = np.mean([d for _, d in G_.degree()])
    clus = nx.transitivity(G_)
    path = nx.average_shortest_path_length(G_) if nx.is_connected(G_) else float("nan")
    print(f"{name:<15} {deg:>12.2f} {clus:>12.3f} {path:>10.2f}")

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (name, G_) in zip(axes, networks.items()):
    pos_ = nx.spring_layout(G_, seed=42)
    nx.draw_networkx(G_, pos=pos_, ax=ax,
                     node_color="#18889F", node_size=200,
                     edge_color="#33333B", with_labels=False,
                     width=1)
    ax.set_title(name)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
n_reps = 20
np.random.seed(3)

results_net = pd.concat([
    diffusion_model(G=G_, p_t=0.15, r_max=50, sim=i).assign(network=name)
    for name, G_ in networks.items()
    for i in range(n_reps)
], ignore_index=True)

mean_net = results_net.groupby(["network", "time"])["proportion"].mean().reset_index()

fig, ax = plt.subplots(figsize=(9, 4))
for net, grp in mean_net.groupby("network"):
    ax.plot(grp["time"], grp["proportion"], label=net)
ax.set_ylim(0, 1.05)
ax.set_xlabel("Time")
ax.set_ylabel("Proportion of adopters")
ax.set_title("Effect of network topology ($p_t=0.15$)")
ax.legend(title="Network")
plt.tight_layout()
plt.show()

The scale-free network typically spreads fastest because hubs act as super-spreaders — once a hub adopts, it can transmit to many neighbours simultaneously. The lattice is slowest because innovations must travel step-by-step through a long chain of connections.

### <span style="color:red">Exercise:</span>

Implement another network layout from this (long) list: https://networkx.org/documentation/stable/reference/generators.html

In [ ]:
N = 200
networks = {
    "random":      nx.erdos_renyi_graph(n=N, p=6/(N-1),    seed=42),
    "small-world": nx.watts_strogatz_graph(n=N, k=6, p=0.1, seed=42),
    "scale-free":  nx.barabasi_albert_graph(n=N, m=3,       seed=42),
    "lattice":     nx.watts_strogatz_graph(n=N, k=6, p=0,   seed=42),
    "your layout": )
}

# Print summary statistics
print(f"{'Network':<15} {'Mean degree':>12} {'Clustering':>12} {'Avg path':>10}")
print("-" * 52)
for name, G_ in networks.items():
    deg  = np.mean([d for _, d in G_.degree()])
    clus = nx.transitivity(G_)
    path = nx.average_shortest_path_length(G_) if nx.is_connected(G_) else float("nan")
    print(f"{name:<15} {deg:>12.2f} {clus:>12.3f} {path:>10.2f}")

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, (name, G_) in zip(axes, networks.items()):
    pos_ = nx.spring_layout(G_, seed=42)
    nx.draw_networkx(G_, pos=pos_, ax=ax,
                     node_color="#18889F", node_size=200,
                     edge_color="#33333B", with_labels=False,
                     width=1)
    ax.set_title(name)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
n_reps = 20
np.random.seed(3)

results_net = pd.concat([
    diffusion_model(G=G_, p_t=0.15, r_max=50, sim=i).assign(network=name)
    for name, G_ in networks.items()
    for i in range(n_reps)
], ignore_index=True)

mean_net = results_net.groupby(["network", "time"])["proportion"].mean().reset_index()

fig, ax = plt.subplots(figsize=(9, 4))
for net, grp in mean_net.groupby("network"):
    ax.plot(grp["time"], grp["proportion"], label=net)
ax.set_ylim(0, 1.05)
ax.set_xlabel("Time")
ax.set_ylabel("Proportion of adopters")
ax.set_title("Effect of network topology ($p_t=0.15$)")
ax.legend(title="Network")
plt.tight_layout()
plt.show()

# E. Seeding strategies

So far we have seeded the model with a single random node. In practice, interventions can choose **who** to seed first. Rogers' framework distinguishes between seeding **innovators** (hubs, opinion leaders) vs. random individuals.

We compare three seeding strategies on a scale-free network:
- **Random**: seed a uniformly random node
- **Hub**: seed the highest-degree node
- **Peripheral**: seed the lowest-degree node

In [ ]:
def diffusion_model_seed(G, p_t, r_max, strategy="random", sim=1):
    """Diffusion model with different seeding strategies.

    Parameters
    ----------
    G        (nx.Graph): social network
    p_t      (float)   : transmission probability
    r_max    (int)     : number of rounds
    strategy (str)     : 'random', 'hub'
    sim      (int)     : simulation ID

    Returns
    -------
    pd.DataFrame: columns time, proportion, strategy, sim
    """
    N      = G.number_of_nodes()
    adjm   = nx.to_numpy_array(G, dtype=bool)
    state  = np.zeros(N, dtype=int)

    degrees = np.array([d for _, d in sorted(G.degree())])
    if strategy == "hub":
        seed = int(np.argmax(degrees))
    else:
        seed = np.random.randint(N)

    state[seed] = 1

    rows = []
    for r in range(r_max):
        new_state = state.copy()
        for i in np.where(state == 1)[0]:
            neighbours = np.where(adjm[i])[0]
            for j in neighbours:
                if state[j] == 0 and np.random.uniform() < p_t:
                    new_state[j] = 1
        state = new_state
        rows.append({
            "time":       r + 1,
            "proportion": np.sum(state) / N,
            "strategy":   strategy,
            "sim":        sim
        })

    return pd.DataFrame(rows)

In [ ]:
G_sf   = nx.barabasi_albert_graph(n=200, m=3, seed=42)
n_reps = 30

np.random.seed(4)
results_seed = pd.concat([
    diffusion_model_seed(G=G_sf, p_t=0.1, r_max=40, strategy=s, sim=i)
    for s in ["random", "hub"]
    for i in range(n_reps)
], ignore_index=True)

mean_seed = results_seed.groupby(["strategy", "time"])["proportion"].mean().reset_index()

fig, ax = plt.subplots(figsize=(9, 4))
for strategy, grp in mean_seed.groupby("strategy"):
    ax.plot(grp["time"], grp["proportion"], label=strategy)
ax.set_ylim(0, 1.05)
ax.set_xlabel("Time")
ax.set_ylabel("Proportion of adopters")
ax.set_title("Effect of seeding strategy on a scale-free network ($p_t=0.1$)")
ax.legend(title="Seed")
plt.tight_layout()
plt.show()

Seeding a hub dramatically accelerates early diffusion. This is the basis of **opinion leader** strategies in public health and marketing — targeting well-connected individuals first to kick-start adoption.

Note that the advantage of hub seeding diminishes over time: eventually all strategies reach similar final proportions, because the network is well-connected enough that the innovation spreads regardless of where it starts. The benefit of hub seeding is **speed**, not reach.

### <span style="color:red">Exercise:</span>

Test the effect of transmission probability $p_t$ and abandonment rate $p_a$ on the seeding strategies.

I already modified the `diffusion_model_seed` function to add abandonment.

In [ ]:
def diffusion_model_seed(G, p_t, p_a, r_max, strategy="random", sim=1):
    """Diffusion model with different seeding strategies.

    Parameters
    ----------
    G        (nx.Graph): social network
    p_t      (float)   : transmission probability
    p_a  (float)   : abandonment probability per round
    r_max    (int)     : number of rounds
    strategy (str)     : 'random', 'hub'
    sim      (int)     : simulation ID

    Returns
    -------
    pd.DataFrame: columns time, proportion, strategy, sim
    """
    N      = G.number_of_nodes()
    adjm   = nx.to_numpy_array(G, dtype=bool)
    state  = np.zeros(N, dtype=int)

    degrees = np.array([d for _, d in sorted(G.degree())])
    if strategy == "hub":
        seed = int(np.argmax(degrees))
    else:
        seed = np.random.randint(N)

    state[seed] = 1
    rows = []
    for r in range(r_max):
        new_state = state.copy()
        for i in range(N):
            if state[i] == 1:
                # Abandonment
                if np.random.uniform() < p_a:
                    new_state[i] = 0
            else:
                # Transmission from adopting neighbours
                neighbours = np.where(adjm[i])[0]
                n_adopters = np.sum(state[neighbours])
                if n_adopters > 0:
                    # At least one adopting neighbour: attempt transmission
                    if np.random.uniform() < 1 - (1 - p_t) ** n_adopters:
                        new_state[i] = 1
        state = new_state
        rows.append({
            "time":       r + 1,
            "proportion": np.sum(state) / N,
            "strategy":   strategy,
            "sim":        sim
        })

    return pd.DataFrame(rows)

In [ ]:
G_sf   = nx.barabasi_albert_graph(n=200, m=3, seed=42)
n_reps = 30

np.random.seed(4)
results_seed = pd.concat([
    diffusion_model_seed(G=G_sf, p_t=1, p_a=0.05, r_max=40, strategy=s, sim=i)
    for s in ["random", "hub"]
    for i in range(n_reps)
], ignore_index=True)

mean_seed = results_seed.groupby(["strategy", "time"])["proportion"].mean().reset_index()

fig, ax = plt.subplots(figsize=(9, 4))
for strategy, grp in mean_seed.groupby("strategy"):
    ax.plot(grp["time"], grp["proportion"], label=strategy)
ax.set_ylim(0, 1.05)
ax.set_xlabel("Time")
ax.set_ylabel("Proportion of adopters")
ax.set_title("Effect of seeding strategy on a scale-free network ($p_t=0.1$)")
ax.legend(title="Seed")
plt.tight_layout()
plt.show()